# Stage 40 — Ratings & credit transitions

**Owner:** Risk Analyst / Credit Risk  
**Pipeline stage:** `pipelines.stage_40_transitions`

Assign credit ratings and attach a transition matrix. **arpym concept #1 lives here**: if a `migration_events` artifact exists, the annual transition matrix is estimated with the rigorous continuous-time generator + entropy-regularised monotonicity estimator (`fit_trans_matrix_credit`); otherwise the engine's baseline matrix is used.

**Reads:** `persons_scored`, optional `migration_events`  
**Writes:** `transition_matrix`, `ratings`, `rating_distribution`

> This notebook is *modular*: it only needs its upstream artifacts to exist in the
> shared store. You can re-run just this stage without touching the others.


## 1. Setup

In [ ]:
# --- setup: make the project root importable + open the shared artifact store ---
import sys, os
ROOT = os.path.abspath("..")          # notebooks/ lives one level below the project root
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from pipelines import ArtifactStore
store = ArtifactStore("output/etl")   # the SAME store every stage reads/writes
print("artifact store:", store.root)
print("existing artifacts:", store.list())


## 2. Run this stage

In [ ]:
from pipelines.stage_40_transitions import run as run_transitions

# OPTIONAL — supply real cohort migration events to trigger the arpym #1 estimator:
#   import pandas as pd
#   events = pd.DataFrame({"period":[...], "from_state":[...],
#                          "to_state":[...], "count":[...]})
#   store.write_df("migration_events", events)
result = run_transitions(store, estimate_from_events=True, tau_hl_years=2.0)
print(result.summary())


## 3. Inspect what this stage produced

In [ ]:
import numpy as np
P = store.read_array("transition_matrix")
print("transition matrix rows sum to 1:", np.allclose(P.sum(axis=1), 1.0))
print("default-column PD by rating:", np.round(P[:7, -1], 4))
display(store.read_df("ratings").head())
print("rating_distribution:", store.read_json("rating_distribution"))
